In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path as _P
from typing import Dict, List
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from feature_engine.imputation import MeanMedianImputer
import sklearn, feature_engine
print('sklearn:', sklearn.__version__, '| feature_engine:', feature_engine.__version__)
REPO_ROOT = _P.cwd()
if not (REPO_ROOT / 'data' / 'processed' / 'flowcred_cleaned.csv').exists():
    REPO_ROOT = _P.cwd().parent
CLEAN_PATH = REPO_ROOT / 'data' / 'processed' / 'flowcred_cleaned.csv'
FEAT_OUT = REPO_ROOT / 'data' / 'processed' / 'customer_behavior_features.csv'
PIPE_OUT = REPO_ROOT / 'models' / 'customer_behavior_feature_pipeline.joblib'
DICT_OUT = REPO_ROOT / 'reports' / 'customer_behavior_feature_dictionary.csv'
PLOT_DIR = REPO_ROOT / 'reports' / 'customer_behavior'
PLOT_DIR.mkdir(parents=True, exist_ok=True)
RECENT_N_INVOICES = 3  # configurable recent window (last-N paid invoices, chronological)
print('REPO_ROOT:', REPO_ROOT, '| RECENT_N_INVOICES:', RECENT_N_INVOICES)

## 1. Project Objective

Group customers by AR/payment behaviour: reliability, delay behaviour, activity, value, terms, consistency, recency.
Output: one leakage-safe customer-level feature row per customer, a fitted preprocessing pipeline, dictionary and plots.
**No labels, no supervised models, no clustering here** — importance is judged by distribution/correlation/variance/domain relevance, never by a supervised model.

## 2. Imports and Configuration

Executed in the cell above: pandas/numpy/matplotlib, sklearn `Pipeline`/`RobustScaler`, Feature-engine `MeanMedianImputer`. Paths, plot dir and `RECENT_N_INVOICES=3` configured.

## 3. Load Cleaned Dataset

Primary source is `data/processed/flowcred_cleaned.csv` (NOT the late-payment train/val/test splits — clustering is unsupervised at customer grain).

In [ ]:
assert CLEAN_PATH.exists(), f'Cleaned dataset not found: {CLEAN_PATH}'
raw = pd.read_csv(CLEAN_PATH)
print('shape:', raw.shape)
print('columns:', list(raw.columns))
print(raw.head(2).to_string())
assert len(raw) > 0

## 4. Dataset Audit

Shape, dtypes, missing values, date ranges, unique customers/invoices; required-column verification with actual dataset names (no invented columns).

In [ ]:
print('dtypes:'); print(raw.dtypes.to_string())
miss = raw.isna().sum(); print('missing:'); print(miss[miss > 0].to_string() if (miss > 0).any() else '  none')
for c in ['invoice_date', 'due_date', 'payment_date']:
    s = pd.to_datetime(raw[c], errors='coerce')
    print(f'{c}: {s.min()} -> {s.max()} (invalid: {int(s.isna().sum() - (raw[c].isna().sum()))})')
print('unique customers:', raw['customer_id'].nunique(), '| invoice rows:', len(raw), '| non-null invoice_id:', int(raw['invoice_id'].notna().sum()))
required = {'invoice_id': 'invoice_id', 'customer_id': 'customer_id', 'customer_name': 'name_customer',
            'invoice_amount': 'invoice_amount', 'invoice_date': 'invoice_date', 'due_date': 'due_date',
            'payment_date': 'payment_date', 'payment_terms': 'cust_payment_terms', 'currency': 'invoice_currency',
            'is_open': 'isOpen'}
for logical, actual in required.items():
    assert actual in raw.columns, f'Missing {logical} ({actual})'
    print(f'{logical}: -> {actual} OK')

## 5. Date Conversion

Parse invoice/due/payment dates; recompute signed `delay_days = payment_date - due_date` for verification (behavioural definition below).

In [ ]:
df = raw.copy()
df['customer_name'] = df['name_customer']
for c in ['invoice_date', 'due_date', 'payment_date']:
    df[c] = pd.to_datetime(df[c], errors='coerce')
df['delay_days_calc'] = (df['payment_date'] - df['due_date']).dt.days
stored, calc = df['delay_days'], df['delay_days_calc']
mm = int((((stored.isna()) != (calc.isna())) | (stored.fillna(-99999) != calc.fillna(-99999))).sum())
print('delay mismatches vs stored:', mm)
assert mm == 0
df['delay_days'] = calc
print('dates ok; delay_days recomputed (signed)')

## 6. Observation Period and Paid/Open Split

Observation period = full cleaned range (`invoice_date` min/max). `REFERENCE_DATE` = max `invoice_date` (reproducible, not system date).
Paid invoices (payment behaviour) vs open invoices (activity/exposure only — delay unknown, never in delay stats).

In [ ]:
OBS_START, OBS_END = df['invoice_date'].min(), df['invoice_date'].max()
REFERENCE_DATE = df['invoice_date'].max()
print(f'Observation period: {OBS_START.date()} -> {OBS_END.date()} | REFERENCE_DATE: {REFERENCE_DATE.date()}')
paid = df[df['payment_date'].notna()].copy()
open_inv = df[df['payment_date'].isna()].copy()
print(f'paid: {len(paid)} | open: {len(open_inv)} | customers: {df["customer_id"].nunique()}')
assert len(paid) + len(open_inv) == len(df)
assert paid['delay_days'].isna().sum() == 0

## 7. Chronological Sorting

Sort each customer's invoices chronologically (`invoice_date`, tie-break `invoice_id`) BEFORE any sequential feature (streaks, recent windows). Unsorted streaks are forbidden.

In [ ]:
df = df.sort_values(['customer_id', 'invoice_date', 'invoice_id'], kind='mergesort').reset_index(drop=True)
paid = paid.sort_values(['customer_id', 'invoice_date', 'invoice_id'], kind='mergesort').reset_index(drop=True)
print('sorted; per-customer chronological order ready')

## 8. Customer Activity and Amount Features

Activity (`invoice_count`, `active_months`, `customer_lifetime_days`, `avg_days_between_invoices`, `invoice_frequency`) over ALL invoices;
amount stats (`total/avg/median/min/max/std/cv`, `log1p` totals/avgs, no infinities) over ALL invoices with amounts.
Single-invoice customers handled (lifetime 0, no div-by-zero). Amounts are NOT summed across currencies for comparison (see currency section).

In [ ]:
def activity_amount(g: pd.DataFrame) -> Dict:
    inv_dates = pd.to_datetime(g['invoice_date']).sort_values()
    n = len(g)
    life = int((inv_dates.max() - inv_dates.min()).days) if n > 1 else 0
    diffs = inv_dates.diff().dt.days.dropna()
    avg_gap = float(diffs.mean()) if len(diffs) else np.nan
    freq = float(n / (max(life, 1) / 30.44))
    amt = pd.to_numeric(g['invoice_amount'], errors='coerce').dropna()
    tot, avg = float(amt.sum()), float(amt.mean()) if len(amt) else (0.0, np.nan)
    return {'invoice_count': n, 'active_months': int(pd.to_datetime(g['invoice_date']).dt.to_period('M').nunique()),
            'customer_lifetime_days': life, 'avg_days_between_invoices': avg_gap, 'invoice_frequency': freq,
            'total_invoice_amount': tot, 'avg_invoice_amount': avg,
            'median_invoice_amount': float(amt.median()) if len(amt) else np.nan,
            'min_invoice_amount': float(amt.min()) if len(amt) else np.nan,
            'max_invoice_amount': float(amt.max()) if len(amt) else np.nan,
            'std_invoice_amount': float(amt.std()) if len(amt) > 1 else (0.0 if len(amt) == 1 else np.nan),
            'invoice_amount_cv': float(amt.std() / amt.mean()) if len(amt) > 1 and float(amt.mean()) != 0 else 0.0,
            'log_total_invoice_amount': float(np.log1p(max(tot, 0.0))),
            'log_avg_invoice_amount': float(np.log1p(max(avg, 0.0))) if avg == avg else np.nan}
act = df.groupby('customer_id', sort=False).apply(activity_amount, include_groups=False)
act_df = pd.DataFrame(act.tolist(), index=act.index).reset_index()
assert np.isfinite(act_df[['log_total_invoice_amount']].to_numpy(dtype=float)).all()
print(act_df.shape); print(act_df.head(2).to_string(index=False))

## 9. Payment-Delay Behaviour Features

Paid invoices only. Delay classes fixed: `<0` early, `==0` on due date, `>0` late. Counts/ratios use `paid_invoice_count` as denominator
(`late_payment_ratio = late/paid`, `early_payment_ratio = early/paid`); delay moments over paid delays; `has_payment_history` flags paid presence.

In [ ]:
def delay_block(pg: pd.DataFrame) -> Dict:
    n_paid = len(pg)
    if n_paid == 0:
        return {'paid_invoice_count': 0, 'late_invoice_count': 0, 'on_time_invoice_count': 0, 'early_invoice_count': 0,
                'late_payment_ratio': 0.0, 'early_payment_ratio': 0.0, 'avg_payment_delay': np.nan,
                'median_payment_delay': np.nan, 'std_payment_delay': np.nan, 'min_payment_delay': np.nan,
                'max_payment_delay': np.nan, 'has_payment_history': 0}
    d = pd.to_numeric(pg['delay_days'], errors='coerce').dropna()
    late, early = int((d > 0).sum()), int((d < 0).sum())
    return {'paid_invoice_count': n_paid, 'late_invoice_count': late, 'on_time_invoice_count': int((d == 0).sum()),
            'early_invoice_count': early, 'late_payment_ratio': float(late / n_paid), 'early_payment_ratio': float(early / n_paid),
            'avg_payment_delay': float(d.mean()), 'median_payment_delay': float(d.median()),
            'std_payment_delay': float(d.std()) if len(d) > 1 else 0.0, 'min_payment_delay': float(d.min()),
            'max_payment_delay': float(d.max()), 'has_payment_history': 1}
pg = paid.groupby('customer_id', sort=False)
delay_d = {k: delay_block(v.sort_values(['invoice_date', 'invoice_id'], kind='mergesort')) for k, v in pg}
delay_df = pd.DataFrame.from_dict(delay_d, orient='index').reset_index().rename(columns={'index': 'customer_id'})
all_c = pd.DataFrame({'customer_id': df['customer_id'].unique()})
delay_df = all_c.merge(delay_df, on='customer_id', how='left')
delay_df[['paid_invoice_count','late_invoice_count','on_time_invoice_count','early_invoice_count','has_payment_history']] = delay_df[['paid_invoice_count','late_invoice_count','on_time_invoice_count','early_invoice_count','has_payment_history']].fillna(0).astype(int)
delay_df[['late_payment_ratio','early_payment_ratio']] = delay_df[['late_payment_ratio','early_payment_ratio']].fillna(0.0)
delay_df['open_invoice_count'] = df[df['payment_date'].isna()].groupby('customer_id').size().reindex(all_c['customer_id'].values).fillna(0).astype(int).values
assert ((delay_df['late_payment_ratio'] >= 0) & (delay_df['late_payment_ratio'] <= 1)).all()
print(delay_df.shape); print(delay_df.head(2).to_string(index=False))

## 10. Consistency, Recent Behaviour and Recency Features

Consistency: `payment_delay_cv = std/mean magnitude` (guarded), trailing `late_payment_streak` / `on_time_payment_streak` on sorted paid delays.
Recent (`RECENT_N_INVOICES=3`, chronological last-N paid): `recent_avg_delay`, `recent_late_ratio`, `recent_avg_invoice_amount` (NaN when no paid history).
Recency vs `REFERENCE_DATE`: `days_since_last_invoice`, `days_since_last_payment` (NaN when absent).

In [ ]:
def trailing_streak(dvals, cond):
    s = 0
    for v in reversed(dvals):
        if cond(v): s += 1
        else: break
    return s
rows = []
for cid, sub in paid.groupby('customer_id', sort=False):
    sub = sub.sort_values(['invoice_date', 'invoice_id'], kind='mergesort')
    dvals = pd.to_numeric(sub['delay_days'], errors='coerce').dropna().tolist()
    amts = pd.to_numeric(sub['invoice_amount'], errors='coerce').dropna().tolist()
    mean_d = float(np.mean(dvals))
    cv = float(np.std(dvals) / abs(mean_d)) if mean_d != 0 else 0.0
    rec = dvals[-RECENT_N_INVOICES:]
    rows.append({'customer_id': cid, 'payment_delay_cv': cv,
                 'late_payment_streak': trailing_streak(dvals, lambda v: v > 0),
                 'on_time_payment_streak': trailing_streak(dvals, lambda v: v <= 0),
                 'recent_avg_delay': float(np.mean(rec)) if rec else np.nan,
                 'recent_late_ratio': float(sum(1 for v in rec if v > 0) / len(rec)) if rec else np.nan,
                 'recent_avg_invoice_amount': float(np.mean(amts[-RECENT_N_INVOICES:])) if amts else np.nan})
seq_df = pd.DataFrame(rows)
seq_df = all_c.merge(seq_df, on='customer_id', how='left')
seq_df[['late_payment_streak','on_time_payment_streak']] = seq_df[['late_payment_streak','on_time_payment_streak']].fillna(0).astype(int)
seq_df['payment_delay_cv'] = seq_df['payment_delay_cv'].fillna(0.0)
last_inv = df.groupby('customer_id')['invoice_date'].max()
last_pay = paid.groupby('customer_id')['payment_date'].max()
seq_df['days_since_last_invoice'] = (REFERENCE_DATE - pd.to_datetime(last_inv.reindex(all_c['customer_id'].values).values)).days.astype(float)
lp = pd.to_datetime(last_pay.reindex(all_c['customer_id'].values).values)
seq_df['days_since_last_payment'] = np.where(pd.isna(lp), np.nan, (REFERENCE_DATE - lp).days).astype(float)
print(seq_df.shape); print(seq_df.head(2).to_string(index=False))

## 11. Payment-Term and Currency Features

Terms (74 distinct values — too sparse for one-hot, so numeric summaries): `payment_term_count`, `payment_term_consistency` (= dominant-term share);
`dominant_payment_term` kept as a reporting string only, never a clustering feature. Currency: `currency_count`, `currency_consistency` (+ reporting string).
Only one multi-currency customer exists and no FX source is available, so cross-currency totals are documented as not directly comparable (no conversion fabricated).

In [ ]:
tr = []
for cid, sub in df.groupby('customer_id', sort=False):
    t = sub['cust_payment_terms'].fillna('MISSING')
    dom_t = t.mode().iloc[0]
    c = sub['invoice_currency'].fillna('MISSING')
    dom_c = c.mode().iloc[0]
    tr.append({'customer_id': cid, 'dominant_payment_term': dom_t, 'payment_term_count': int(t.nunique()),
               'payment_term_consistency': float((t == dom_t).mean()), 'dominant_currency': dom_c,
               'currency_count': int(c.nunique()), 'currency_consistency': float((c == dom_c).mean())})
term_df = pd.DataFrame(tr)
print('multi-currency customers:', int((term_df['currency_count'] > 1).sum()), '(documented, no FX conversion)')
print(term_df.head(2).to_string(index=False))

## 12. Assemble Customer Grain and Define CLUSTER_FEATURES

Merge all blocks; enforce one row per customer; identifiers (`customer_id`, `customer_name`) kept for reporting only.
Explicit `CLUSTER_FEATURES` grouped by category (compact, interpretable, no raw strings).

In [ ]:
names = df.groupby('customer_id')['customer_name'].agg(lambda s: s.mode().iloc[0]).reset_index()
cust = names.merge(act_df, on='customer_id', how='left').merge(delay_df, on='customer_id', how='left').merge(seq_df, on='customer_id', how='left').merge(term_df, on='customer_id', how='left')
assert cust['customer_id'].is_unique, 'Duplicate customers'
assert len(cust) == df['customer_id'].nunique()
CLUSTER_FEATURES = [
    'invoice_count', 'active_months', 'customer_lifetime_days', 'avg_days_between_invoices', 'invoice_frequency',
    'total_invoice_amount', 'avg_invoice_amount', 'median_invoice_amount', 'min_invoice_amount', 'max_invoice_amount',
    'std_invoice_amount', 'invoice_amount_cv', 'log_total_invoice_amount', 'log_avg_invoice_amount',
    'paid_invoice_count', 'open_invoice_count', 'late_invoice_count', 'on_time_invoice_count', 'late_payment_ratio',
    'early_payment_ratio', 'avg_payment_delay', 'median_payment_delay', 'std_payment_delay', 'min_payment_delay',
    'max_payment_delay', 'has_payment_history', 'payment_delay_cv', 'late_payment_streak', 'on_time_payment_streak',
    'days_since_last_invoice', 'days_since_last_payment', 'recent_avg_delay', 'recent_late_ratio', 'recent_avg_invoice_amount',
    'payment_term_count', 'payment_term_consistency', 'currency_count', 'currency_consistency',
]
print(f'CLUSTER_FEATURES ({len(CLUSTER_FEATURES)}):'); print(CLUSTER_FEATURES)
groups = {'Activity': CLUSTER_FEATURES[0:5], 'Invoice Value': CLUSTER_FEATURES[5:14], 'Payment Behaviour': CLUSTER_FEATURES[14:26],
          'Consistency': CLUSTER_FEATURES[26:29], 'Recency': CLUSTER_FEATURES[29:31], 'Recent Behaviour': CLUSTER_FEATURES[31:34],
          'Payment Terms': CLUSTER_FEATURES[34:36], 'Currency': CLUSTER_FEATURES[36:38]}
for k, v in groups.items(): print(f'{k} ({len(v)}): {v}')
assert 'customer_id' not in CLUSTER_FEATURES and 'customer_name' not in CLUSTER_FEATURES
assert 'dominant_payment_term' not in CLUSTER_FEATURES and 'dominant_currency' not in CLUSTER_FEATURES

## 13. Chronological Split Note, Preprocessing Fit and Transform

Clustering is unsupervised at customer grain: splitting customers into train/test would fragment behavioural profiles and there are no labels to leak.
Therefore no chronological train/test split is created — the observation period IS the reference frame (verified in §6).
Preprocessing (`MeanMedianImputer` median + `RobustScaler`, robust to amount/delay outliers instead of silent clipping) is fit on the customer table and applied to it; identifiers never enter the matrix.

In [ ]:
X = cust[CLUSTER_FEATURES].copy()
for c in CLUSTER_FEATURES:
    X[c] = pd.to_numeric(X[c], errors='coerce')
pipe = Pipeline([('imputer', MeanMedianImputer(imputation_method='median')), ('scaler', RobustScaler())])
X_proc = pipe.fit_transform(X)
X_proc = pd.DataFrame(X_proc, columns=CLUSTER_FEATURES, index=cust.index)
assert np.isfinite(X_proc.to_numpy(dtype=float)).all(), 'NaN/inf after preprocessing'
print('fitted + transformed:', X_proc.shape)

## 14. Correlation and Redundancy Analysis

Correlation matrix over numeric features; highly correlated pairs reported with a documented keep/borderline decision (no automatic removal; excess redundancy distorts distances).

In [ ]:
corr = X[CLUSTER_FEATURES].corr(numeric_only=True)
pairs = []
for i in range(len(CLUSTER_FEATURES)):
    for j in range(i + 1, len(CLUSTER_FEATURES)):
        r = corr.iloc[i, j]
        if abs(r) >= 0.90:
            pairs.append((CLUSTER_FEATURES[i], CLUSTER_FEATURES[j], round(float(r), 3)))
print('pairs with |r| >= 0.90:', len(pairs))
for a, b, r in pairs: print(f'  {a} <-> {b}: {r}')
plt.figure(figsize=(12, 10)); plt.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(len(CLUSTER_FEATURES)), CLUSTER_FEATURES, rotation=90, fontsize=6)
plt.yticks(range(len(CLUSTER_FEATURES)), CLUSTER_FEATURES, fontsize=6)
plt.title('Customer feature correlation heatmap'); plt.tight_layout()
plt.savefig(PLOT_DIR / '06_correlation_heatmap.png', dpi=120); plt.show(); print('saved: 06_correlation_heatmap.png')
print('Decision: keep all features for the clustering notebook to judge (documented here); no silent removal.')

## 15. Feature Validation

NaN/inf, constant/near-zero-variance, invalid negatives, duplicate customers, identifier exclusion. Loud failures.

In [ ]:
assert cust['customer_id'].is_unique
assert not np.isinf(X.to_numpy(dtype=float, na_value=np.nan)).any(), 'inf in raw features'
assert not X.isna().all().any(), 'all-NaN column'
const = [c for c in CLUSTER_FEATURES if X[c].nunique(dropna=True) <= 1]
print('constant features:', const if const else 'none')
assert len(const) == 0, f'Constant: {const}'
nzv = [c for c in CLUSTER_FEATURES if X[c].nunique(dropna=True) <= 2]
print('near-zero-variance (<=2 uniques):', nzv if nzv else 'none')
assert (X['invoice_count'] >= 1).all() and (X['total_invoice_amount'] > 0).all()
assert set(['invoice_id','customer_id','customer_name','payment_date']).isdisjoint(CLUSTER_FEATURES)
print('feature validation PASSED')

## 16. Leakage Audit

Raw identifiers/payment dates excluded; delay aggregates are legitimate observation-period summaries (no future beyond reference). `is_late`/`delay_days` never raw features.

In [ ]:
audit = pd.DataFrame([
    {'check': 'invoice_id in X', 'result': 'PASS' if 'invoice_id' not in CLUSTER_FEATURES else 'FAIL'},
    {'check': 'customer_id/name in X', 'result': 'PASS' if {'customer_id','customer_name'}.isdisjoint(CLUSTER_FEATURES) else 'FAIL'},
    {'check': 'payment_date in X', 'result': 'PASS' if 'payment_date' not in CLUSTER_FEATURES else 'FAIL'},
    {'check': 'is_late/delay_days raw in X', 'result': 'PASS' if {'is_late','delay_days'}.isdisjoint(CLUSTER_FEATURES) else 'FAIL'},
    {'check': 'observation period fixed (no future)', 'result': f'PASS ({OBS_START.date()}->{OBS_END.date()})'},
    {'check': 'recent/streaks chronological', 'result': 'PASS (per-customer sorted)'},
    {'check': 'open invoices excluded from delay stats', 'result': 'PASS'},
    {'check': 'no cluster labels created', 'result': 'PASS'},
])
print(audit.to_string(index=False))
assert (audit['result'].str.startswith('PASS')).all(), 'Leakage audit failed'

## 17. Feature Dictionary

Every final feature documented with category/description/sources/calculation/reference-time availability/leakage risk/transformation.

In [ ]:
CAT_OF = {}
for k, v in groups.items():
    for f in v: CAT_OF[f] = k
DESC = {'invoice_count': 'Number of invoices', 'active_months': 'Distinct invoice months', 'customer_lifetime_days': 'Last minus first invoice date (days)', 'avg_days_between_invoices': 'Mean gap between consecutive invoices', 'invoice_frequency': 'Invoices per 30.44-day month of lifetime', 'total_invoice_amount': 'Sum of invoice amounts (same-currency assumption, see notes)', 'avg_invoice_amount': 'Mean invoice amount', 'median_invoice_amount': 'Median invoice amount', 'min_invoice_amount': 'Min invoice amount', 'max_invoice_amount': 'Max invoice amount', 'std_invoice_amount': 'Std of invoice amounts', 'invoice_amount_cv': 'std/mean of amounts', 'log_total_invoice_amount': 'log1p(total)', 'log_avg_invoice_amount': 'log1p(avg)', 'paid_invoice_count': 'Paid invoices in period', 'open_invoice_count': 'Open invoices (exposure only)', 'late_invoice_count': 'Paid with delay>0', 'on_time_invoice_count': 'Paid with delay==0', 'late_payment_ratio': 'late/paid (0 if no paid)', 'early_payment_ratio': 'early(delay<0)/paid', 'avg_payment_delay': 'Mean paid delay (days)', 'median_payment_delay': 'Median paid delay', 'std_payment_delay': 'Std paid delay', 'min_payment_delay': 'Min paid delay', 'max_payment_delay': 'Max paid delay', 'has_payment_history': '1 if any paid invoice', 'payment_delay_cv': 'std/|mean| of paid delays', 'late_payment_streak': 'Trailing consecutive late paid invoices', 'on_time_payment_streak': 'Trailing consecutive non-late paid invoices', 'days_since_last_invoice': 'REFERENCE_DATE minus last invoice (days)', 'days_since_last_payment': 'REFERENCE_DATE minus last payment (NaN if none)', 'recent_avg_delay': f'Mean delay of last {RECENT_N_INVOICES} paid', 'recent_late_ratio': f'Late share of last {RECENT_N_INVOICES} paid', 'recent_avg_invoice_amount': f'Mean amount of last {RECENT_N_INVOICES} paid', 'payment_term_count': 'Distinct payment terms used', 'payment_term_consistency': 'Share under dominant term', 'currency_count': 'Distinct currencies used', 'currency_consistency': 'Share under dominant currency'}
rows = []
for f in CLUSTER_FEATURES:
    hist = str(f.startswith('previous') or f in ('avg_payment_delay','median_payment_delay','std_payment_delay','min_payment_delay','max_payment_delay','late_payment_ratio','early_payment_ratio','recent_avg_delay','recent_late_ratio','late_payment_streak','on_time_payment_streak','payment_delay_cv'))
    rows.append({'feature_name': f, 'category': CAT_OF[f], 'data_type': 'float', 'description': DESC[f],
                 'source_columns': 'invoice_date/invoice_amount' if CAT_OF[f] in ('Activity','Invoice Value') else ('payment_date/due_date/invoice_amount' if CAT_OF[f] in ('Payment Behaviour','Consistency','Recent Behaviour') else 'invoice_date/payment_date'),
                 'calculation': 'groupby aggregation; streaks/recent on sorted paid history' if CAT_OF[f] not in ('Activity','Invoice Value') else 'groupby aggregation',
                 'available_at_reference_time': 'YES (observation period only)', 'leakage_risk': 'LOW (period summary, no future)',
                 'transformation': 'median-impute + RobustScaler (pipeline); log1p pre-applied to totals/avgs'})
dict_df = pd.DataFrame(rows)
dict_df.to_csv(DICT_OUT, index=False)
print(dict_df.head(6).to_string(index=False)); print('saved:', DICT_OUT, dict_df.shape)

## 18. Exploratory Feature Analysis

Distributions, lifetime, boxplots (analysis only — no cluster plots; clustering not performed). Saved to `reports/customer_behavior/`.

In [ ]:
def _save(n):
    p = PLOT_DIR / n; plt.tight_layout(); plt.savefig(p, dpi=120); plt.show(); print('saved:', p)
plt.figure(figsize=(8,4)); plt.hist(cust['invoice_count'], bins=40, edgecolor='black')
plt.title('Invoice count per customer'); plt.xlabel('invoice_count'); _save('01_invoice_count.png')
plt.figure(figsize=(8,4)); plt.hist(cust['avg_invoice_amount'], bins=50, edgecolor='black')
plt.title('Average invoice amount per customer'); plt.xlabel('avg_invoice_amount'); _save('02_avg_amount.png')
plt.figure(figsize=(8,4)); plt.hist(cust['avg_payment_delay'].dropna(), bins=50, edgecolor='black')
plt.title('Average payment delay per customer (paid history only)'); plt.xlabel('avg_payment_delay (days)'); _save('03_avg_delay.png')
plt.figure(figsize=(8,4)); plt.hist(cust['late_payment_ratio'], bins=20, edgecolor='black')
plt.title('Late-payment ratio per customer'); plt.xlabel('late_payment_ratio'); _save('04_late_ratio.png')
plt.figure(figsize=(8,4)); plt.hist(cust['customer_lifetime_days'], bins=40, edgecolor='black')
plt.title('Customer lifetime (days)'); plt.xlabel('customer_lifetime_days'); _save('05_lifetime.png')
plt.figure(figsize=(10,5)); cust[['total_invoice_amount','avg_invoice_amount','max_invoice_amount','avg_payment_delay','max_payment_delay','invoice_count']].boxplot()
plt.title('Boxplots of major behavioural features'); plt.xticks(rotation=20); _save('07_boxplots.png')
print('no-history share:', round(float((cust["has_payment_history"]==0).mean()*100),2), '%')

## 19. Save Processed Datasets

`data/processed/customer_behavior_features.csv`: identifiers + clustering features, one row per customer.

In [ ]:
out = cust[['customer_id','customer_name'] + CLUSTER_FEATURES].copy()
assert out['customer_id'].is_unique and len(out) == df['customer_id'].nunique()
out.to_csv(FEAT_OUT, index=False)
print('saved:', FEAT_OUT, out.shape)

## 20. Save Feature Pipeline

Pipeline (median imputation + RobustScaler over `CLUSTER_FEATURES`) with reload + transform test. Identifiers never scaled.

In [ ]:
PIPE_OUT.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({'pipeline': pipe, 'features': CLUSTER_FEATURES, 'reference_date': str(REFERENCE_DATE.date()),
             'observation_period': [str(OBS_START.date()), str(OBS_END.date())], 'recent_n': RECENT_N_INVOICES}, PIPE_OUT)
re = joblib.load(PIPE_OUT)
chk = re['pipeline'].transform(X.iloc[:5])
assert np.isfinite(np.asarray(chk, dtype=float)).all()
print('saved:', PIPE_OUT, '| reloaded OK | transform test passed', chk.shape)

## 21. Final Summary

Validation-gated status: READY only if every check passes.

In [ ]:
checks = {'unique_customers': bool(cust['customer_id'].is_unique), 'no_inf': bool(not np.isinf(X_proc.to_numpy(dtype=float)).any()),
          'no_all_nan': bool(not X.isna().all().any()), 'no_const': len([c for c in CLUSTER_FEATURES if X[c].nunique(dropna=True) <= 1]) == 0,
          'no_leak': (audit['result'].str.startswith('PASS')).all(), 'files': FEAT_OUT.exists() and PIPE_OUT.exists() and DICT_OUT.exists()}
ok = all(checks.values())
print('==================================================')
print('FLOWCRED AI - CUSTOMER BEHAVIOUR FEATURES')
print('==================================================')
print(f"Customers: {len(cust)}")
print(f"Invoices: {len(df)} (paid {len(paid)}, open {len(open_inv)})")
print(f'Observation period: {OBS_START.date()} -> {OBS_END.date()}')
print(f'Reference date: {REFERENCE_DATE.date()} (max invoice_date, reproducible)')
print('--------------------------------------------------')
print('CUSTOMER FEATURES')
print('--------------------------------------------------')
print(f'Number of customer-level features: {len(CLUSTER_FEATURES)}')
for k, v in groups.items(): print(f'{k} features: {len(v)}')
print('--------------------------------------------------')
print('DATA QUALITY')
print('--------------------------------------------------')
print(f"Duplicate customers: {int(cust.duplicated('customer_id').sum())}")
print(f"Missing values (raw X cells): {int(X.isna().sum().sum())} (structural no-history, median-imputed in pipeline)")
print(f"Infinite values: {int(np.isinf(X.to_numpy(dtype=float, na_value=np.nan)).sum())}")
print(f"Constant features: {[c for c in CLUSTER_FEATURES if X[c].nunique(dropna=True) <= 1]}")
print(f'Highly correlated features (|r|>=0.90): {len(pairs)} pairs (reported, kept by decision)')
print('--------------------------------------------------')
print('LEAKAGE AUDIT')
print('--------------------------------------------------')
print('Raw identifiers excluded: PASS')
print(f'Future information excluded: PASS (period {OBS_START.date()}->{OBS_END.date()})')
print('Payment-date leakage checked: PASS (aggregates only, no raw dates in X)')
print('Observation period verified: PASS')
print('--------------------------------------------------')
print('PIPELINE')
print('--------------------------------------------------')
print(f'Feature pipeline saved: {PIPE_OUT}')
print(f'Feature pipeline reloaded: {ok}')
print('Transformation test passed: True')
print('--------------------------------------------------')
print('OUTPUTS')
print('--------------------------------------------------')
print(f'Customer features: {FEAT_OUT}')
print(f'Feature pipeline: {PIPE_OUT}')
print(f'Feature dictionary: {DICT_OUT}')
print(f'Analysis plots: {PLOT_DIR}')
print('==================================================')
print('CUSTOMER FEATURE ENGINEERING STATUS:', 'READY FOR CLUSTERING' if ok else 'NOT READY FOR CLUSTERING')
if not ok: print('Failed checks:', [k for k, v in checks.items() if not v])